# 02 — Preparación de Datos

## Objetivo

Construir datasets supervisados reproducibles (train/validation/test) para forecasting temporal, garantizando ausencia de leakage.

## Inputs
- `data/raw/daily_withdrawals.csv`
- `manifests/01_data_understanding.json`

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from datetime import datetime

PROJECT_ROOT = Path('.').resolve()
if PROJECT_ROOT.name != 'de-junior-tecnico-a-senior-de-negocio':
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'daily_withdrawals.csv'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
MANIFESTS_DIR = PROJECT_ROOT / 'manifests'

# Validar fase anterior
prev_manifest = json.load(open(MANIFESTS_DIR / '01_data_understanding.json'))
assert prev_manifest['status'] == 'GO', f"Fase anterior no aprobada: {prev_manifest['status']}"
print(f"✅ Fase anterior: {prev_manifest['status']}")


✅ Fase anterior: GO


## 1. Carga y ordenamiento cronológico

In [2]:
df = pd.read_csv(RAW_PATH, parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)
print(f"Shape: {df.shape}")
print(f"Periodo: {df['date'].min().date()} a {df['date'].max().date()}")


Shape: (730, 11)
Periodo: 2024-07-01 a 2026-06-30


## 2. Features de calendario

Todas las features de calendario de D+1 son válidas (se conocen de antemano).

In [3]:
# Agregar month como feature adicional
df['month'] = df['date'].dt.month
print("Features de calendario disponibles:")
cal_features = ['day_of_week', 'is_weekend', 'is_holiday', 'is_payday', 'is_month_end', 'days_to_payday', 'month']
for f in cal_features:
    print(f"  {f}: {df[f].nunique()} valores únicos")


Features de calendario disponibles:
  day_of_week: 7 valores únicos
  is_weekend: 2 valores únicos
  is_holiday: 2 valores únicos
  is_payday: 2 valores únicos
  is_month_end: 2 valores únicos
  days_to_payday: 16 valores únicos
  month: 12 valores únicos


## 3. Lags y ventanas móviles

**Regla crítica:** Todo lag/rolling debe usar `shift(1)` para NO incluir información del día D+1.

Patrón correcto: `series.shift(1).rolling(window).mean()`

In [4]:
TARGET = 'total_withdrawals_cop'

# Lags: información estrictamente pasada
df['lag_1'] = df[TARGET].shift(1)
df['lag_7'] = df[TARGET].shift(7)
df['lag_14'] = df[TARGET].shift(14)
df['lag_28'] = df[TARGET].shift(28)

# Rolling: shift(1) ANTES del rolling para excluir día actual
shifted = df[TARGET].shift(1)
df['rolling_mean_7'] = shifted.rolling(7, min_periods=7).mean()
df['rolling_mean_14'] = shifted.rolling(14, min_periods=14).mean()
df['rolling_std_7'] = shifted.rolling(7, min_periods=7).std()

print("Features temporales creadas:")
temporal_features = ['lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_14', 'rolling_std_7']
for f in temporal_features:
    nulls = df[f].isnull().sum()
    print(f"  {f}: {nulls} NaN (warm-up)")


Features temporales creadas:
  lag_1: 1 NaN (warm-up)
  lag_7: 7 NaN (warm-up)
  lag_14: 14 NaN (warm-up)
  lag_28: 28 NaN (warm-up)
  rolling_mean_7: 7 NaN (warm-up)
  rolling_mean_14: 14 NaN (warm-up)
  rolling_std_7: 7 NaN (warm-up)


## 4. Validación de leakage

**Pregunta:** ¿Alguna feature contiene información del futuro?

In [5]:
# Test de leakage: para cada fila i, verificar que los lags
# NO usen el valor de la fila i (que es el target a predecir)
print("=== Validación de leakage ===")

# lag_1[i] debe ser target[i-1], NO target[i]
for i in range(1, min(10, len(df))):
    assert df['lag_1'].iloc[i] == df[TARGET].iloc[i-1], f"Leakage en lag_1, fila {i}"
print("✅ lag_1: correcto (usa valor de i-1)")

# lag_7[i] debe ser target[i-7]
for i in range(7, min(20, len(df))):
    assert df['lag_7'].iloc[i] == df[TARGET].iloc[i-7], f"Leakage en lag_7, fila {i}"
print("✅ lag_7: correcto (usa valor de i-7)")

# rolling_mean_7[i] NO debe incluir target[i]
# rolling_mean_7[i] = mean(target[i-7:i]) con shift, es decir mean(target[i-8:i-1])
# Verificamos que rolling_mean_7[i] != incluyendo target[i]
for i in range(28, min(40, len(df))):
    rolling_val = df['rolling_mean_7'].iloc[i]
    target_val = df[TARGET].iloc[i]
    window_with_current = df[TARGET].iloc[i-6:i+1].mean()  # si incluyera i
    window_without = df[TARGET].iloc[i-7:i].mean()  # correcto: shift(1) + rolling(7)
    assert abs(rolling_val - window_without) < 1, f"Leakage en rolling, fila {i}"
print("✅ rolling_mean_7: correcto (excluye día actual)")

# El target NUNCA es feature
feature_cols = [c for c in df.columns if c not in ['date', TARGET]]
assert TARGET not in feature_cols, "LEAKAGE: target es feature!"
print(f"✅ Target '{TARGET}' NO está en features")
print(f"\nTotal features: {len(feature_cols)}")
print(f"Features: {feature_cols}")


=== Validación de leakage ===
✅ lag_1: correcto (usa valor de i-1)
✅ lag_7: correcto (usa valor de i-7)
✅ rolling_mean_7: correcto (excluye día actual)
✅ Target 'total_withdrawals_cop' NO está en features

Total features: 17
Features: ['transaction_count', 'day_of_week', 'is_weekend', 'is_holiday', 'is_payday', 'is_month_end', 'days_to_payday', 'trend', 'special_event', 'month', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_14', 'rolling_std_7']


## 5. Separación temporal (train / validation / test)

NUNCA random split. El test se BLOQUEA hasta evaluation-business.

In [6]:
# Eliminar filas con NaN (warm-up de 28 días por lag_28)
df_clean = df.dropna().reset_index(drop=True)
print(f"Filas después de eliminar warm-up: {len(df_clean)} (de {len(df)})")

# Separación: 60% train, 20% validation, 20% test
n = len(df_clean)
train_end = int(n * 0.6)
val_end = int(n * 0.8)

train_df = df_clean.iloc[:train_end].copy()
val_df = df_clean.iloc[train_end:val_end].copy()
test_df = df_clean.iloc[val_end:].copy()

print(f"\n=== Separación temporal ===")
print(f"Train:      {len(train_df)} días ({train_df['date'].min().date()} a {train_df['date'].max().date()})")
print(f"Validation: {len(val_df)} días ({val_df['date'].min().date()} a {val_df['date'].max().date()})")
print(f"Test:       {len(test_df)} días ({test_df['date'].min().date()} a {test_df['date'].max().date()})")

# Verificar que no hay overlap
assert train_df['date'].max() < val_df['date'].min(), "Overlap train-val!"
assert val_df['date'].max() < test_df['date'].min(), "Overlap val-test!"
print("\n✅ Sin overlap entre conjuntos")


Filas después de eliminar warm-up: 702 (de 730)

=== Separación temporal ===
Train:      421 días (2024-07-29 a 2025-09-22)
Validation: 140 días (2025-09-23 a 2026-02-09)
Test:       141 días (2026-02-10 a 2026-06-30)

✅ Sin overlap entre conjuntos


## 6. Documentación de features

In [7]:
feature_docs = pd.DataFrame({
    'Feature': feature_cols,
    'Tipo': ['calendario' if f in cal_features else 'lag' if 'lag' in f else 'rolling' if 'rolling' in f else 'contexto' for f in feature_cols],
    'Disponible_en_D': ['Si'] * len(feature_cols),
    'Usa_shift': ['N/A' if f in cal_features + ['transaction_count', 'trend', 'special_event'] else 'Si' for f in feature_cols]
})
print(feature_docs.to_string(index=False))


          Feature       Tipo Disponible_en_D Usa_shift
transaction_count   contexto              Si       N/A
      day_of_week calendario              Si       N/A
       is_weekend calendario              Si       N/A
       is_holiday calendario              Si       N/A
        is_payday calendario              Si       N/A
     is_month_end calendario              Si       N/A
   days_to_payday calendario              Si       N/A
            trend   contexto              Si       N/A
    special_event   contexto              Si       N/A
            month calendario              Si       N/A
            lag_1        lag              Si        Si
            lag_7        lag              Si        Si
           lag_14        lag              Si        Si
           lag_28        lag              Si        Si
   rolling_mean_7    rolling              Si        Si
  rolling_mean_14    rolling              Si        Si
    rolling_std_7    rolling              Si        Si


## 7. Exportación de datasets

In [8]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

train_df.to_csv(PROCESSED_DIR / 'train.csv', index=False)
val_df.to_csv(PROCESSED_DIR / 'validation.csv', index=False)
test_df.to_csv(PROCESSED_DIR / 'test.csv', index=False)

print(f"✅ Guardado: data/processed/train.csv ({len(train_df)} filas)")
print(f"✅ Guardado: data/processed/validation.csv ({len(val_df)} filas)")
print(f"✅ Guardado: data/processed/test.csv ({len(test_df)} filas)")

# Feature list para reproducibilidad
feature_list = {'features': feature_cols, 'target': TARGET}
with open(PROCESSED_DIR / 'feature_list.json', 'w') as f:
    json.dump(feature_list, f, indent=2)
print(f"✅ Guardado: data/processed/feature_list.json")


✅ Guardado: data/processed/train.csv (421 filas)
✅ Guardado: data/processed/validation.csv (140 filas)
✅ Guardado: data/processed/test.csv (141 filas)
✅ Guardado: data/processed/feature_list.json


## 8. Conclusiones y decisión

### Estado: **GO** ✅

### Resumen
- 17 features construidas sin leakage.
- Separación cronológica train/validation/test (60/20/20).
- Test set bloqueado hasta evaluation-business.
- Todos los lags usan shift correcto.

### Siguiente paso
Agente: **modeling-tournament**

In [9]:
# Generar manifest
manifest = {
    "phase": "data-preparation",
    "status": "GO",
    "started_at": datetime.now().isoformat(),
    "completed_at": datetime.now().isoformat(),
    "inputs": ["data/raw/daily_withdrawals.csv", "manifests/01_data_understanding.json"],
    "outputs": [
        "notebooks/02_data_preparation.ipynb",
        "data/processed/train.csv",
        "data/processed/validation.csv",
        "data/processed/test.csv",
        "data/processed/feature_list.json",
        "reports/02_data_preparation.md",
        "manifests/02_data_preparation.json"
    ],
    "decisions": [
        "Usar shift(1) en todos los lags y rolling",
        "Separacion 60/20/20 cronologica",
        "lag_28 incluido para capturar patrones mensuales",
        "Test bloqueado hasta evaluation-business"
    ],
    "metrics": {
        "total_features": len(feature_cols),
        "train_rows": len(train_df),
        "validation_rows": len(val_df),
        "test_rows": len(test_df),
        "warmup_rows_lost": len(df) - len(df_clean)
    },
    "assumptions": [
        "60/20/20 es adecuado para 702 dias utiles",
        "28 dias de warm-up es aceptable",
        "transaction_count es del dia D, no D+1"
    ],
    "risks": [
        "Tendencia puede afectar desempeno en test (mas reciente)",
        "lag_28 pierde mas filas por warm-up"
    ],
    "tests_executed": ["leakage_lag1", "leakage_lag7", "leakage_rolling", "target_not_feature", "no_overlap_splits"],
    "human_approval_required": True,
    "human_approved": False,
    "next_agent": "modeling-tournament"
}

with open(MANIFESTS_DIR / '02_data_preparation.json', 'w') as f:
    json.dump(manifest, f, indent=2)
print("✅ Manifest guardado: manifests/02_data_preparation.json")


✅ Manifest guardado: manifests/02_data_preparation.json
